In [13]:
!pip install transformers==4.39.3 accelerate==0.27.2 peft==0.10.0 bitsandbytes datasets -q

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import torch
import json

In [15]:
dataset_path = "/content/augmented_dataset_v1.json"
with open(dataset_path, "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples")
print("Sample:", data[0])

train_data = [
    {
        "input_text": f"Generate 3 domain names for this business. Business: {item['business_description']}\nDomains:",
        "target_text": ", ".join(item['expected_domain_names'])
    }
    for item in data
]

dataset = Dataset.from_list(train_data)
dataset = dataset.train_test_split(test_size=0.1)
print(dataset)

Loaded 3000 samples
Sample: {'business_description': 'startup for pets in Singapore', 'expected_domain_names': ['startuppets.biz', 'startuppets.io', 'singaporeio.co']}
DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 2700
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 300
    })
})


In [16]:
model_name = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",           # Auto GPU placement
    torch_dtype=torch.float16,   # Half precision
    load_in_4bit=True            # 4-bit quantization
)
print("Loaded Phi-2 in 4-bit for LoRA fine-tuning")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded Phi-2 in 4-bit for LoRA fine-tuning


In [22]:
from peft import prepare_model_for_kbit_training

# ✅ Prepare model for k-bit training before LoRA
model = prepare_model_for_kbit_training(model)

# ✅ Apply LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 5,242,880 || all params: 2,784,926,720 || trainable%: 0.1882591725788749


In [23]:
def tokenize(batch):
    inputs = tokenizer(batch["input_text"], padding="max_length", truncation=True, max_length=128)
    targets = tokenizer(batch["target_text"], padding="max_length", truncation=True, max_length=128)
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["input_text", "target_text"])
print(tokenized_dataset)

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2700
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 300
    })
})


In [25]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# Enable gradient checkpointing for lower memory usage
model.gradient_checkpointing_enable()

In [29]:
training_args = TrainingArguments(
    output_dir="./phi2-lora-finetuned",
    per_device_train_batch_size=1,       # Safe for Colab T4
    gradient_accumulation_steps=8,       # Simulate larger batch
    learning_rate=2e-4,
    num_train_epochs=2,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",            # Memory-efficient optimizer
    report_to="none"
)

In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

trainer.train()

Step,Training Loss
10,0.554000
20,0.581200
30,0.577300
40,0.549400
50,0.521700
60,0.501200
70,0.480800
80,0.488300
90,0.482700
100,0.470800


TrainOutput(global_step=674, training_loss=0.44926075075783434, metrics={'train_runtime': 3086.4026, 'train_samples_per_second': 1.75, 'train_steps_per_second': 0.218, 'total_flos': 1.098976101138432e+16, 'train_loss': 0.44926075075783434, 'epoch': 2.0})

In [31]:
model.save_pretrained("./phi2_aug_lora")
tokenizer.save_pretrained("./phi2_aug_lora")
print("Model saved!")

Model saved!


In [32]:
!zip -r phi2_aug_lora.zip /content/phi2_aug_lora

  adding: content/phi2_aug_lora/ (stored 0%)
  adding: content/phi2_aug_lora/added_tokens.json (deflated 84%)
  adding: content/phi2_aug_lora/adapter_config.json (deflated 52%)
  adding: content/phi2_aug_lora/merges.txt (deflated 53%)
  adding: content/phi2_aug_lora/special_tokens_map.json (deflated 75%)
  adding: content/phi2_aug_lora/tokenizer.json (deflated 72%)
  adding: content/phi2_aug_lora/adapter_model.safetensors (deflated 8%)
  adding: content/phi2_aug_lora/tokenizer_config.json (deflated 94%)
  adding: content/phi2_aug_lora/vocab.json (deflated 59%)
  adding: content/phi2_aug_lora/README.md (deflated 66%)
